In [1]:
from pypdf import PdfReader

In [2]:

from pypdf import PdfReader

# Read the PDF
reader = PdfReader("dummyData.pdf")
text = "\n".join(page.extract_text() or "" for page in reader.pages)

# Chunk the document
chunks = [chunk.strip() for chunk in text.split("\n\n") if chunk.strip()]

# If chunks are too few, split by sentence
if len(chunks) < 5:
    chunks = [s.strip() + "." for s in text.split(".") if s.strip()]

print(f"Created {len(chunks)} chunks")

chunks


Created 6 chunks


['Space Exploration Knowledge Base \x7f RAG Demo Dataset\nPage 1\n SPACE EXPLORATION\n KNOWLEDGE BASE\n A document collection for building and testing a Retrieval-Augmented Generation (RAG)\n application\nDataset theme: planets, the Moon, Mars missions, spacecraft, black holes, exoplanets, and life-support\nsystems.\nPurpose\nThis PDF is designed as a small, self-contained knowledge source for a demonstration project. It contains short,\nclearly separated sections with facts and explanations that can be split into chunks, embedded, stored in a vector\ndatabase, and retrieved in response to natural-language questions.\nHow to use it\nUse this file as the source document for your RAG pipeline. Ask questions that match a specific section, ask the\nsame question using different wording, and ask questions whose answers are not included to test whether your\napplication admits uncertainty.\nImportant note\nThis is an educational dataset, not an official mission manual. It summarizes general 

In [5]:
from sentence_transformers import SentenceTransformer

#load the sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [6]:
#generate embeddings for all chunks
embeddings = model.encode(chunks, show_progress_bar=True)
embeddings, len(embeddings)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

(array([[ 0.00147229,  0.04468763,  0.04032221, ..., -0.01481785,
         -0.10931608,  0.02418917],
        [ 0.00142208,  0.00743993,  0.07521439, ...,  0.02467453,
         -0.01150907,  0.01332306],
        [-0.02852239,  0.01277911,  0.03635597, ...,  0.03817028,
         -0.08324551, -0.04671793],
        [-0.00890914, -0.07216478,  0.00721915, ..., -0.07860494,
         -0.01263234,  0.02403962],
        [-0.02542546, -0.00722117,  0.04999496, ...,  0.03149679,
         -0.01726929,  0.03128922],
        [-0.03975961, -0.01390774,  0.03808178, ..., -0.00020891,
         -0.02568687,  0.03301794]], dtype=float32),
 6)

In [7]:
import faiss
import numpy as np
import pickle

#create faiss index
dimension = embeddings.shape[1] #384 for all-MiniLM-L6-v2
index = faiss.IndexFlatL2(dimension)

#add embeddings to the index
index.add(np.array(embeddings).astype('float32'))

print(f"FAISS index created with {index.ntotal} vectors")
print(f"Embeddings dimension: {dimension}")
#save faiss index to diskk
faiss.write_index(index, 'dummyData_index.faiss')

#save the chunks for later retrieval
with open('dummyDatachunks.pkl', 'wb') as f:
          pickle.dump(chunks, f)

print("Index and chunks saved successfuly")


FAISS index created with 6 vectors
Embeddings dimension: 384
Index and chunks saved successfuly


In [8]:
def search_policy(query, top_k=2):
          """
          Perform similarity search on FAISS index with a user query.

          Args:
                    query (str): The user question or search query
                    top_k (int): Number of most similar chunks to retrieve

          Returns:
                   list: List of tuples containing (chunk_text, distance)
                   """
          #Encode the query using the same model
          query_embedding = model.encode([query])

          #search the FAISS index
          distances, indices = index.search(np.array(query_embedding).astype('float32'), top_k)

          #retrieve the corresponding chunks
          results = []
          for i, (idx,distance) in enumerate(zip(indices[0], distances[0])):
                  results.append((chunks[idx], distance))
          return results

#example usage
user_query = "What is the document all about?"

results = search_policy(user_query, top_k=3)

print(f"Query: {user_query}\n")
print("Top matching policy sections:\n")

for i, (chunk, distance) in enumerate(results, 1):
        print(f"{i}. [Distance: {distance:.4f}]")
        print(chunk)
        print("-" * 50)


Query: What is the document all about?

Top matching policy sections:

1. [Distance: 1.3918]
Space Exploration Knowledge Base  RAG Demo Dataset
Page 6
 What are the main differences between asteroids and comets?
 Why do we see nearly the same side of the Moon?
 What makes landing a heavy spacecraft on Mars difficult?
 How does a rocket generate thrust in space?
 Explain the difference between a probe, a lander, and a rover.
 How can scientists detect a black hole if it does not emit light?
 What is the difference between an exoplanet and a planet in our Solar System?
 Does the habitable zone prove that a planet has life?
 Why do astronauts need exercise in microgravity?
 Why is communication with a distant spacecraft delayed?
 What is a gravity assist and why is it useful?
 What risks are created by space debris?
 What is the return period for products in this knowledge base?
 Which rocket engine model is used for the fictional Orion-X mission?
 Evaluation note: The fina

In [9]:
results

[('Space Exploration Knowledge Base \x7f RAG Demo Dataset\nPage 6\n\x7f What are the main differences between asteroids and comets?\n\x7f Why do we see nearly the same side of the Moon?\n\x7f What makes landing a heavy spacecraft on Mars difficult?\n\x7f How does a rocket generate thrust in space?\n\x7f Explain the difference between a probe, a lander, and a rover.\n\x7f How can scientists detect a black hole if it does not emit light?\n\x7f What is the difference between an exoplanet and a planet in our Solar System?\n\x7f Does the habitable zone prove that a planet has life?\n\x7f Why do astronauts need exercise in microgravity?\n\x7f Why is communication with a distant spacecraft delayed?\n\x7f What is a gravity assist and why is it useful?\n\x7f What risks are created by space debris?\n\x7f What is the return period for products in this knowledge base?\n\x7f Which rocket engine model is used for the fictional Orion-X mission?\n Evaluation note: The final two questions are intention

In [11]:
%pip install groq

  Using cached groq-1.7.0-py3-none-any.whl.metadata (16 kB)
  Using cached distro-1.9.0-py3-none-any.whl.metadata (6.8 kB)
Using cached groq-1.7.0-py3-none-any.whl (143 kB)
Using cached distro-1.9.0-py3-none-any.whl (20 kB)

   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   ---------------------------------------- 0/2 [distro]
   -------------------------------


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
import os #to read an environment variable containing your API key.
from dotenv import load_dotenv
from groq import Groq

load_dotenv() #loads variables from your .env file.

api_key = os.getenv("GROQ_API_KEY")

model_name = "openai/gpt-oss-120b" #model identifier
TOP_P = 0.9
Temperature = 0.2

def model_request(prompt_template: str, system_message: str, model_name: str, **kwargs) -> str:

    client = Groq(api_key=api_key)

    response = client.chat.completions.create(
        model=model_name,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": prompt_template}
        ],
        top_p=kwargs.get("top_p", TOP_P),
        temperature=kwargs.get("temperature", Temperature)
    )

    return response.choices[0].message.content


#prompt = "What is the capital of France?"

prompt_template = f"""
                Based on the following context, answer the question.
                Context:
                {results}
                Question: what is the document all about?
                """


system_message = ("You are a helpful assistant that provides accurate information.")

response_text = model_request(prompt_template, system_message, model_name)

print(response_text)


The document is a small, self‑contained “Space Exploration Knowledge Base” that was created as a demo dataset for Retrieval‑Augmented Generation (RAG) applications. It gathers concise, topic‑focused sections on a range of space‑science subjects—planets, the Moon, Mars missions, spacecraft design, black holes, exoplanets, life‑support systems, space debris, gravity assists, and related quick‑fact Q&A. Its purpose is to serve as a test collection for building and evaluating RAG pipelines, showing how a vector‑store can retrieve relevant facts and how a system should respond (including admitting uncertainty when a question falls outside the provided material).
